# Pipeline di Preprocessing: Approcci Giornaliero e Settimanale 🇮🇹 [🇬🇧](03_preprocessing.ipynb)

## Previsione del Rischio di Infortunio nei Runner — Replica di Lovdal et al. (2021)

Questo notebook implementa e documenta la **pipeline di preprocessing** per entrambi gli approcci temporali (giornaliero e settimanale). Trasforma i dati raw caricati in dataset puliti e suddivisi, pronti per la modellazione.

### Passaggi della pipeline

1. **Sostituzione sentinel** (solo giornaliero): −0.01 → 0.0 per le metriche percettive nei giorni di riposo (ADR-007)
2. **Split train/test**: GroupShuffleSplit per ID atleta — nessun leakage tra atleti (ADR-006)
3. **Binarizzazione del target** (solo settimanale): punteggio infortunio continuo → binario alla soglia 0.5 (ADR-002)
4. **Scaling**: StandardScaler adattato solo sui dati di training — salvato per uso condizionale nella modellazione
5. **Persistenza**: split processati e scaler salvati in `data/processed/`

### Decisioni progettuali chiave

- **Salvataggio dati non scalati**: i modelli tree-based (RF, XGBoost) non necessitano di scaling. Gli scaler sono salvati separatamente per la Logistic Regression.
- **Formato Parquet**: preserva i dtype, file più piccoli, I/O più veloce rispetto al CSV.
- **Class weighting rispetto a SMOTE**: strategia primaria per lo sbilanciamento (ADR-003).

Ogni passaggio si collega a un risultato dell'EDA dai notebook 01–02 e a una decisione ADR.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ATHLETE_ID_COL,
    INJURY_COL,
    N_DAY_BLOCKS,
    PROCESSED_DATA_DIR,
    RANDOM_SEED,
    SENTINEL_VALUE,
    WEEK_INJURY_THRESHOLD,
)
from src.data_loading import get_feature_columns, load_day_data, load_week_data
from src.preprocessing.common import (
    get_feature_target_groups,
    get_group_kfold,
    split_train_test,
)
from src.preprocessing.day_preprocessor import (
    fit_scaler,
    handle_sentinel_values,
    transform_scaled,
)
from src.preprocessing.io import load_scaler, load_splits, save_scaler, save_splits
from src.preprocessing.week_preprocessor import binarize_target
from src.utils.logging_config import setup_logging
from src.utils.plotting import INJURY_PALETTE, PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

## 1. Caricamento Dati

Carichiamo entrambi i dataset raw usando le stesse funzioni validate dalla fase di EDA. La rinomina delle colonne e la validazione dello schema avvengono automaticamente all'interno di `load_day_data()` e `load_week_data()`.

In [ ]:
df_day = load_day_data()
df_week = load_week_data()

day_feature_cols = get_feature_columns(df_day)
week_feature_cols = get_feature_columns(df_week)

print("Day approach:")
print(f"  Shape: {df_day.shape[0]:,} rows × {df_day.shape[1]} columns")
print(f"  Feature columns: {len(day_feature_cols)}")
print("\nWeek approach:")
print(f"  Shape: {df_week.shape[0]:,} rows × {df_week.shape[1]} columns")
print(f"  Feature columns: {len(week_feature_cols)}")

---

## 2. Trattamento del Valore Sentinel (Approccio Giornaliero)

Il dataset dell'approccio giornaliero usa **−0.01** come valore sentinel per le metriche percettive (`perceived_exertion`, `perceived_training_success`, `perceived_recovery`) nei giorni di riposo — giorni in cui l'atleta non si è allenato e quindi non aveva valutazioni soggettive da riportare.

**Perché sostituire con 0.0?** (ADR-007): se un atleta non si è allenato, il suo sforzo percepito è zero (nessuno sforzo), il suo carico di recupero è zero (niente da cui recuperare). Il sentinel −0.01 introdurrebbe un segnale artificiale nelle operazioni di scaling e basate sulla distanza.

L'approccio settimanale **non** ha valori sentinel — i giorni di riposo sono catturati dalla feature `nr_rest_days` e da valori più bassi negli altri aggregati settimanali.

In [ ]:
# Count sentinel values before replacement
perceived_features = ["perceived_exertion", "perceived_training_success", "perceived_recovery"]
sentinel_counts = {}

for feat in perceived_features:
    cols = [f"day_{b}_{feat}" for b in range(N_DAY_BLOCKS)]
    total = (df_day[cols] == SENTINEL_VALUE).sum().sum()
    pct = total / (df_day.shape[0] * len(cols)) * 100
    sentinel_counts[feat] = {"count": total, "pct": f"{pct:.1f}%"}

sentinel_summary = pd.DataFrame(sentinel_counts).T
sentinel_summary.columns = ["Total sentinel cells", "% of cells"]
print("Sentinel value (−0.01) prevalence BEFORE replacement:\n")
print(sentinel_summary.to_string())

In [ ]:
# Save a copy of one column BEFORE replacement for the comparison plot
before_col = "day_0_perceived_exertion"
before_data = df_day[before_col].copy()

# Apply sentinel replacement
df_day = handle_sentinel_values(df_day)

# Verify: zero sentinels remain
remaining = (df_day[day_feature_cols] == SENTINEL_VALUE).sum().sum()
assert remaining == 0, f"Expected 0 sentinels after replacement, found {remaining}"
print(f"Sentinel replacement complete. Remaining sentinel values: {remaining}")

In [ ]:
# Before/after comparison: day_0_perceived_exertion
after_data = df_day[before_col]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax1.hist(before_data, bins=50, color=PALETTE["neutral"], alpha=0.7, edgecolor="white")
ax1.axvline(SENTINEL_VALUE, color=PALETTE["negative"], linestyle="--", linewidth=1.5,
            label=f"Sentinel ({SENTINEL_VALUE})")
ax1.set_xlabel("Value")
ax1.set_ylabel("Count")
ax1.set_title("BEFORE — with sentinel −0.01", fontweight="bold")
ax1.legend(fontsize=9)

ax2.hist(after_data, bins=50, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
ax2.set_xlabel("Value")
ax2.set_title("AFTER — sentinel replaced with 0.0", fontweight="bold")

fig.suptitle("Sentinel Replacement Effect — day_0_perceived_exertion",
             fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()
sns.despine()

save_figure(fig, "03_sentinel_before_after", subdir="preprocessing", close=False)
plt.show()
plt.close(fig)

### Interpretazione

Il picco del sentinel a −0.01 viene assorbito nel bin dello zero in modo pulito. Il resto della distribuzione è invariato — nessuna informazione viene persa o distorta. Il bin dello zero ora rappresenta correttamente sia "nessun allenamento effettuato" (dalle feature di volume) sia "nessuna valutazione soggettiva registrata" (dalle feature percettive), che sono semanticamente equivalenti: **nessun allenamento = carico zero**.

---

## 3. Split Train/Test (GroupShuffleSplit per Atleta)

Il passaggio di preprocessing più critico: dividere i dati **preservando i confini degli atleti** (ADR-006).

**Perché GroupShuffleSplit?** Se lo stesso atleta appare sia nel training che nel test set, il modello potrebbe apprendere baseline specifiche dell'atleta (es. "l'atleta X ha sempre uno sforzo elevato") anziché predittori di infortunio generalizzabili. GroupShuffleSplit garantisce che tutte le osservazioni di un atleta appaiano in **esattamente uno split**.

- **Rapporto di split**: 80% train / 20% test (per atleti, non per righe)
- **Variabile di raggruppamento**: `athlete_id`
- **Random state**: fissato a 42 per la riproducibilità

In [ ]:
# Split both datasets
day_train, day_test = split_train_test(df_day)
week_train, week_test = split_train_test(df_week)

for name, train, test in [("Day", day_train, day_test), ("Week", week_train, week_test)]:
    n_train_athletes = train[ATHLETE_ID_COL].nunique()
    n_test_athletes = test[ATHLETE_ID_COL].nunique()
    print(f"{name} approach:")
    print(f"  Train: {len(train):,} rows, {n_train_athletes} athletes")
    print(f"  Test:  {len(test):,} rows, {n_test_athletes} athletes")
    print(f"  Row ratio: {len(train) / (len(train) + len(test)):.1%} / {len(test) / (len(train) + len(test)):.1%}")
    print()

In [ ]:
# === CRITICAL CHECK: No athlete leakage ===
day_train_athletes = set(day_train[ATHLETE_ID_COL].unique())
day_test_athletes = set(day_test[ATHLETE_ID_COL].unique())
assert day_train_athletes.isdisjoint(day_test_athletes), "LEAK: day athletes in both splits!"

week_train_athletes = set(week_train[ATHLETE_ID_COL].unique())
week_test_athletes = set(week_test[ATHLETE_ID_COL].unique())
assert week_train_athletes.isdisjoint(week_test_athletes), "LEAK: week athletes in both splits!"

print("No athlete leakage detected in either approach.")
print(f"  Day — Train athletes: {sorted(day_train_athletes)[:5]}... | Test athletes: {sorted(day_test_athletes)[:5]}...")
print(f"  Week — Train athletes: {sorted(week_train_athletes)[:5]}... | Test athletes: {sorted(week_test_athletes)[:5]}...")

In [ ]:
# Visualization: athlete assignment to train/test (day approach)
athlete_obs = (
    df_day.groupby(ATHLETE_ID_COL)
    .size()
    .reset_index(name="n_obs")
    .sort_values("n_obs", ascending=True)
)
athlete_obs["split"] = athlete_obs[ATHLETE_ID_COL].apply(
    lambda x: "Train" if x in day_train_athletes else "Test"
)

fig, ax = plt.subplots(figsize=(12, 14))

colors = [PALETTE["primary"] if s == "Train" else PALETTE["secondary"]
          for s in athlete_obs["split"]]

ax.barh(range(len(athlete_obs)), athlete_obs["n_obs"], color=colors,
        edgecolor="white", height=0.7, linewidth=0.3)

# Legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=PALETTE["primary"], label=f"Train ({len(day_train_athletes)} athletes)"),
    Patch(facecolor=PALETTE["secondary"], label=f"Test ({len(day_test_athletes)} athletes)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=10)

ax.set_xlabel("Number of observations")
ax.set_ylabel("Athlete (sorted by observation count)")
ax.set_title("Athlete Assignment to Train/Test Split — Day Approach", fontweight="bold")
ax.set_yticks(range(0, len(athlete_obs), 5))
sns.despine()

save_figure(fig, "03_train_test_athlete_assignment", subdir="preprocessing", close=False)
plt.show()
plt.close(fig)

In [ ]:
# Class balance in train vs test
print("Class balance — Day approach:")
for name, split in [("Train", day_train), ("Test", day_test)]:
    rate = split[INJURY_COL].mean() * 100
    n_pos = int(split[INJURY_COL].sum())
    print(f"  {name}: {rate:.2f}% positive (n={n_pos:,} / {len(split):,})")

print("\nClass balance — Week approach (continuous, before binarization):")
for name, split in [("Train", week_train), ("Test", week_test)]:
    mean_score = split[INJURY_COL].mean()
    above_thresh = (split[INJURY_COL] >= WEEK_INJURY_THRESHOLD).mean() * 100
    print(f"  {name}: mean injury score = {mean_score:.4f}, ≥{WEEK_INJURY_THRESHOLD} → {above_thresh:.2f}%")

In [ ]:
# GroupKFold preview: how the 5 CV folds partition training athletes
k_fold = get_group_kfold()
X_day_train, y_day_train, groups_day_train = get_feature_target_groups(
    day_train, day_feature_cols
)

fold_info = []
for fold_idx, (train_idx, val_idx) in enumerate(
    k_fold.split(X_day_train, y_day_train, groups_day_train)
):
    fold_groups = groups_day_train.iloc[val_idx]
    fold_y = y_day_train.iloc[val_idx]
    fold_info.append({
        "Fold": fold_idx + 1,
        "Train rows": len(train_idx),
        "Val rows": len(val_idx),
        "Val athletes": fold_groups.nunique(),
        "Val injury %": f"{fold_y.mean() * 100:.2f}%",
    })

fold_df = pd.DataFrame(fold_info)
print("GroupKFold (5-fold) — Day training data:\n")
print(fold_df.to_string(index=False))

### Interpretazione

- **Nessun leakage tra atleti**: gli insiemi di atleti di train e test sono completamente disgiunti.
- **Il rapporto delle righe varia**: poiché GroupShuffleSplit divide per atleti (non per righe), il rapporto effettivo delle righe dipende da quanti dati ha contribuito ogni atleta. Gli atleti con più osservazioni hanno un impatto maggiore sulle proporzioni dello split.
- **Bilanciamento delle classi preservato**: il tasso di infortunio nel train e nel test dovrebbe essere approssimativamente simile. Piccole differenze sono attese a causa dello splitting a livello di atleta — se un atleta con alto tasso di infortuni finisce nel test set, ne sposta il tasso.
- **Anteprima GroupKFold**: i 5 fold partizionano gli atleti di training in gruppi di validazione. Il tasso di infortunio varia tra i fold perché gli atleti differiscono nella suscettibilità agli infortuni — questo è atteso e riflette l'eterogeneità del mondo reale.

Il test set è **tenuto da parte** fino alla valutazione finale del modello (notebook 04/05). Durante lo sviluppo, solo la cross-validation GroupKFold sul training set viene usata per selezionare gli iperparametri e confrontare i modelli.

---

## 4. Binarizzazione del Target Settimanale

Il target dell'approccio settimanale è un **punteggio di infortunio continuo** (da 0.0 a ~1.5+) che deve essere binarizzato per la classificazione. Seguendo la convenzione dell'articolo (ADR-002), usiamo **soglia = 0.5**: valori ≥ 0.5 diventano 1 (infortunato), valori < 0.5 diventano 0 (non infortunato).

L'analisi di sensibilità alla soglia nel notebook 02 ha documentato come il bilanciamento delle classi cambia a soglie diverse (0.1, 0.25, 0.5, 0.75, 1.0).

In [ ]:
# Binarize week target in both splits
week_train[INJURY_COL] = binarize_target(week_train[INJURY_COL])
week_test[INJURY_COL] = binarize_target(week_test[INJURY_COL])

print(f"Week target binarized at threshold {WEEK_INJURY_THRESHOLD}:\n")
for name, split in [("Train", week_train), ("Test", week_test)]:
    rate = split[INJURY_COL].mean() * 100
    n_pos = int(split[INJURY_COL].sum())
    print(f"  {name}: {rate:.2f}% positive (n={n_pos:,} / {len(split):,})")

# Verify binary values only
assert set(week_train[INJURY_COL].unique()).issubset({0, 1})
assert set(week_test[INJURY_COL].unique()).issubset({0, 1})
print("\nTarget is binary (0/1 only) — verified.")

In [ ]:
# Side-by-side class balance comparison: day vs week
day_pos_pct = day_train[INJURY_COL].mean() * 100
week_pos_pct = week_train[INJURY_COL].mean() * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3.5))

for ax, pos_pct, title in [
    (ax1, day_pos_pct, "Day Approach"),
    (ax2, week_pos_pct, "Week Approach"),
]:
    neg_pct = 100 - pos_pct
    labels = [f"No Injury ({neg_pct:.2f}%)", f"Injury ({pos_pct:.2f}%)"]
    colors = [INJURY_PALETTE[0], INJURY_PALETTE[1]]
    bars = ax.barh(labels, [neg_pct, pos_pct], color=colors, height=0.5, edgecolor="white")
    ax.set_xlabel("% of training observations")
    ax.set_title(title, fontweight="bold")
    ax.set_xlim(0, 110)
    for bar, pct in zip(bars, [neg_pct, pos_pct]):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                f"{pct:.2f}%", va="center", fontweight="bold", fontsize=9)

fig.suptitle("Class Balance Comparison (Training Sets)", fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()
sns.despine(left=True)

save_figure(fig, "03_class_balance_comparison", subdir="preprocessing", close=False)
plt.show()
plt.close(fig)

### Interpretazione

Entrambi gli approcci mostrano **sbilanciamento estremo delle classi**, ma l'approccio settimanale è un po' meno grave grazie alla soglia di binarizzazione che cattura una definizione più ampia di infortunio.

Entrambi useranno il **class weighting** come strategia primaria (ADR-003):
- Logistic Regression / Random Forest: `class_weight='balanced'`
- XGBoost: `scale_pos_weight = n_negativi / n_positivi`

Questo dice al modello di penalizzare più pesantemente gli infortuni mal classificati, in proporzione alla loro rarità — una scelta progettuale cruciale quando la classe minoritaria è quella che ci interessa effettivamente prevedere.

---

## 5. Strategia di Scaling

Lo scaling delle feature è **dipendente dal modello**:

| Modello                   | Necessita scaling? | Perché                                                                                    |
|---------------------------|-------------------|-------------------------------------------------------------------------------------------|
| **Logistic Regression**   | Sì                | I coefficienti sono sensibili alla magnitudine; la regolarizzazione L2 assume feature standardizzate |
| **Random Forest**         | No                | Gli split sono basati sull'ordine (confronti di rango), non sulla magnitudine              |
| **XGBoost**               | No                | Come RF — gli split degli alberi sono invarianti a trasformazioni monotone delle feature    |

**Il nostro approccio**: adattare `StandardScaler` solo sulle feature di training (per prevenire il data leakage), salvare lo scaler adattato su disco e applicarlo **condizionalmente** nei notebook di modellazione — solo per la Logistic Regression. I file Parquet salvati contengono dati **non scalati**.

In [ ]:
# Fit scalers on training features only
day_scaler = fit_scaler(day_train[day_feature_cols])
week_scaler = fit_scaler(week_train[week_feature_cols])

print("Day scaler — fitted on training features:")
print(f"  Features: {len(day_scaler.mean_)}")
print(f"  Mean range: [{day_scaler.mean_.min():.4f}, {day_scaler.mean_.max():.4f}]")
print(f"  Scale range: [{day_scaler.scale_.min():.4f}, {day_scaler.scale_.max():.4f}]")
print("\nWeek scaler — fitted on training features:")
print(f"  Features: {len(week_scaler.mean_)}")
print(f"  Mean range: [{week_scaler.mean_.min():.4f}, {week_scaler.mean_.max():.4f}]")
print(f"  Scale range: [{week_scaler.scale_.min():.4f}, {week_scaler.scale_.max():.4f}]")

In [ ]:
# Demonstrate scaling effect: before/after for 3 representative features
demo_features = ["day_0_total_km", "day_0_perceived_exertion", "day_0_km_z3_4"]
day_train_scaled = transform_scaled(day_train, day_scaler, day_feature_cols)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for i, feat in enumerate(demo_features):
    # Raw (top row)
    ax_raw = axes[0, i]
    ax_raw.hist(day_train[feat], bins=50, color=PALETTE["neutral"], alpha=0.7, edgecolor="white")
    ax_raw.set_title(f"{feat.replace('day_0_', '')}\n(raw)", fontweight="bold", fontsize=10)
    if i == 0:
        ax_raw.set_ylabel("Count")

    # Scaled (bottom row)
    ax_scaled = axes[1, i]
    ax_scaled.hist(day_train_scaled[feat], bins=50, color=PALETTE["primary"], alpha=0.7, edgecolor="white")
    ax_scaled.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax_scaled.set_title(f"{feat.replace('day_0_', '')}\n(scaled)", fontweight="bold", fontsize=10)
    if i == 0:
        ax_scaled.set_ylabel("Count")

fig.suptitle("Scaling Effect — Raw vs StandardScaler (Day Training Data)",
             fontweight="bold", fontsize=13, y=1.03)
fig.tight_layout()

save_figure(fig, "03_scaling_before_after", subdir="preprocessing", close=False)
plt.show()
plt.close(fig)

In [ ]:
# Verify no data leakage: test set stats should NOT be exactly mean=0, std=1
day_test_scaled = transform_scaled(day_test, day_scaler, day_feature_cols)
test_stats = day_test_scaled[demo_features].describe().loc[["mean", "std"]]
test_stats.columns = [c.replace("day_0_", "") for c in test_stats.columns]

print("Scaled TEST set statistics (should be close to 0/1 but NOT exact):\n")
print(test_stats.round(4).to_string())
print("\n→ Non-exact values confirm the scaler was fit on TRAIN data only (no leakage).")

### Interpretazione

- **Lo scaling centra le feature a 0** e normalizza la loro dispersione a varianza unitaria. Questo mette tutte le feature sulla stessa scala — fondamentale per la Logistic Regression dove coefficienti e penalità di regolarizzazione sono sensibili alla magnitudine.
- **Forma preservata**: lo scaling è una trasformazione lineare — non cambia la forma della distribuzione (asimmetria, zero-inflazione). La forte zero-inflazione in feature come `km_z3_4` resta visibile anche dopo lo scaling.
- **Nessun leakage verificato**: le statistiche del test set sono vicine a media=0 / std=1 ma non esatte, confermando che lo scaler è stato adattato esclusivamente sui dati di training.

---

## 6. Salvataggio Dati Processati

Persistiamo gli split preprocessati e gli scaler adattati in `data/processed/` per l'uso nei notebook di modellazione (04, 05).

| File                 | Formato | Contenuto                                   |
|----------------------|---------|---------------------------------------------|
| `day_train.parquet`  | Parquet | Train giornaliero — sentinel sostituiti, non scalato |
| `day_test.parquet`   | Parquet | Test giornaliero — sentinel sostituiti, non scalato  |
| `week_train.parquet` | Parquet | Train settimanale — target binarizzato, non scalato  |
| `week_test.parquet`  | Parquet | Test settimanale — target binarizzato, non scalato   |
| `day_scaler.pkl`     | joblib  | StandardScaler adattato sulle feature di train giornaliero |
| `week_scaler.pkl`    | joblib  | StandardScaler adattato sulle feature di train settimanale |

**Perché Parquet?** Preserva i dtype esattamente (nessuna conversione int→float), dimensione file minore rispetto al CSV e lettura/scrittura più veloce. **Perché non scalato?** I modelli tree-based non necessitano di scaling — gli scaler sono salvati separatamente per l'uso condizionale con LogReg.

In [ ]:
# Save all splits and scalers
save_splits(day_train, day_test, prefix="day")
save_splits(week_train, week_test, prefix="week")
save_scaler(day_scaler, name="day_scaler")
save_scaler(week_scaler, name="week_scaler")

print(f"All artifacts saved to: {PROCESSED_DATA_DIR}")

In [ ]:
# Round-trip verification: load back and check integrity
day_train_loaded, day_test_loaded = load_splits(prefix="day")
week_train_loaded, week_test_loaded = load_splits(prefix="week")
day_scaler_loaded = load_scaler(name="day_scaler")
week_scaler_loaded = load_scaler(name="week_scaler")

# Shape checks
assert day_train_loaded.shape == day_train.shape, "Day train shape mismatch"
assert day_test_loaded.shape == day_test.shape, "Day test shape mismatch"
assert week_train_loaded.shape == week_train.shape, "Week train shape mismatch"
assert week_test_loaded.shape == week_test.shape, "Week test shape mismatch"

# No athlete leakage in loaded data
assert set(day_train_loaded[ATHLETE_ID_COL].unique()).isdisjoint(
    set(day_test_loaded[ATHLETE_ID_COL].unique())
), "Leakage in loaded day data!"

# Scaler consistency
np.testing.assert_array_almost_equal(day_scaler.mean_, day_scaler_loaded.mean_)
np.testing.assert_array_almost_equal(week_scaler.mean_, week_scaler_loaded.mean_)

print("Round-trip verification passed:")
print(f"  Day train:  {day_train_loaded.shape}")
print(f"  Day test:   {day_test_loaded.shape}")
print(f"  Week train: {week_train_loaded.shape}")
print(f"  Week test:  {week_test_loaded.shape}")
print(f"  Day scaler:  {len(day_scaler_loaded.mean_)} features")
print(f"  Week scaler: {len(week_scaler_loaded.mean_)} features")

---

## 7. Riepilogo: Pipeline di Preprocessing

### Flusso della pipeline

```
Approccio giornaliero:
  data/raw/day_*.csv
    → load_day_data()           # rinomina colonne, valida schema
    → handle_sentinel_values()  # −0.01 → 0.0 (ADR-007)
    → split_train_test()        # GroupShuffleSplit per atleta (ADR-006)
    → fit_scaler(train)         # StandardScaler, solo train
    → salva in data/processed/  # 2 Parquet + 1 scaler pkl

Approccio settimanale:
  data/raw/week_*.csv
    → load_week_data()          # rinomina colonne, valida schema
    → split_train_test()        # GroupShuffleSplit per atleta (ADR-006)
    → binarize_target()         # soglia 0.5 (ADR-002)
    → fit_scaler(train)         # StandardScaler, solo train
    → salva in data/processed/  # 2 Parquet + 1 scaler pkl
```

### Invarianti verificate

1. **Nessun valore sentinel residuo** nei dati giornalieri dopo la sostituzione
2. **Nessun leakage tra atleti** tra gli split train e test (entrambi gli approcci)
3. **Il target settimanale è binario** (solo 0/1) dopo la binarizzazione
4. **Scaler adattato solo sul train** — le statistiche del test set confermano l'assenza di leakage
5. **Persistenza round-trip** — i dati caricati corrispondono esattamente a quelli salvati

### Cosa ricevono i notebook di modellazione

- **DataFrame non scalati** (Parquet) con tutte le colonne (feature + metadati)
- **Scaler adattati** (joblib) da applicare condizionalmente per la Logistic Regression
- Colonne feature estratte tramite `get_feature_columns(df)`
- Gruppi estratti tramite `get_feature_target_groups(df, feature_cols)`

### Prossimi passi

→ **Notebook 04**: Modellazione — approccio giornaliero (Dummy, LogReg, RF, XGBoost con GroupKFold CV)
→ **Notebook 05**: Modellazione — approccio settimanale (stessa pipeline sui dati settimanali)

In [ ]:
# Final check: list saved artifacts with file sizes
print(f"Contents of {PROCESSED_DATA_DIR}:\n")
for f in sorted(PROCESSED_DATA_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:30s} {size_kb:8.1f} KB")